# Chapter 13 — Pooled Cross Sections & Differences-in-Differences
## University of Neuchâtel — Python for Econometrics

Topics:
1. Pooled cross-sectional data
2. DiD: group means method
3. DiD: OLS regression method
4. DiD with controls
5. Log specification

**Format:** One cell per step. Manual calculation first, then statsmodels, then `np.isclose()`.

## 0 — Setup

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import matplotlib.pyplot as plt
import wooldridge as woo
import os

In [ ]:
chapter_dir = os.path.dirname(os.path.abspath('__file__'))
os.chdir(chapter_dir)
output_dir = os.path.join(chapter_dir, 'output')
os.makedirs(output_dir, exist_ok=True)
print('Working directory:', chapter_dir)

## 1 — Load Data: `kielmc`

Kiel & McClain (1995) study of housing prices near a garbage incinerator in North Andover, MA.

- `rprice`: real house price (inflation-adjusted)
- `year`: 1978 (before) or 1981 (after incinerator announced)
- `nearinc`: 1 if house is near the incinerator site, 0 otherwise

**Research question:** Did the announcement of the incinerator reduce prices of nearby homes?

In [ ]:
df = woo.dataWoo('kielmc')
print('Shape:', df.shape)
print(df[['rprice','year','nearinc']].head(10))

In [ ]:
print('Year distribution:')
print(df['year'].value_counts())
print('\nnearinc distribution:')
print(df['nearinc'].value_counts())

In [ ]:
print('\nDescriptive stats:')
print(df[['rprice','nearinc']].describe().round(0))

## 2 — Pooled Cross Sections

**Pooled cross sections** = multiple cross-sections from different years pooled together.
Unlike panel data, the same individuals are NOT necessarily observed in each year.

Key variables to create:
- `y81`: dummy = 1 if observation is from 1981 (after treatment period)
- `nearinc_y81`: interaction = `nearinc × y81` (the DiD term)

### Step 2.1 — Create time dummy y81

In [ ]:
df['y81'] = (df['year'] == 1981).astype(int)
print('Dataset after creating y81 (first 8 rows):')
print(df[['year','y81','nearinc','rprice']].head(8))

In [ ]:
print('Cross-tabulation of nearinc by y81:')
print(pd.crosstab(df['nearinc'], df['y81'], rownames=['nearinc'], colnames=['y81']))

## 3 — DiD via Group Means

**DiD estimator:**

$$\hat{\delta}_{DiD} = (\bar{Y}_{T,1} - \bar{Y}_{T,0}) - (\bar{Y}_{C,1} - \bar{Y}_{C,0})$$

Where:
- $T$ = treated group (nearinc=1), $C$ = control group (nearinc=0)
- subscript 1 = after (1981), subscript 0 = before (1978)

### Step 3.1 — Compute group means manually

In [ ]:
m_n1_78 = df.loc[(df['nearinc']==1)&(df['y81']==0), 'rprice'].mean()
m_n0_78 = df.loc[(df['nearinc']==0)&(df['y81']==0), 'rprice'].mean()
m_n1_81 = df.loc[(df['nearinc']==1)&(df['y81']==1), 'rprice'].mean()
m_n0_81 = df.loc[(df['nearinc']==0)&(df['y81']==1), 'rprice'].mean()
print('Group means:')
print(f'  Near incinerator, 1978: ${m_n1_78:,.0f}')
print(f'  Not near,          1978: ${m_n0_78:,.0f}')
print(f'  Near incinerator, 1981: ${m_n1_81:,.0f}')
print(f'  Not near,          1981: ${m_n0_81:,.0f}')

### Step 3.2 — Compute DiD manually

In [ ]:
change_near    = m_n1_81 - m_n1_78
change_not_near= m_n0_81 - m_n0_78
DiD_manual = change_near - change_not_near
print(f'Change for near group:     ${change_near:+,.0f}')
print(f'Change for control group:  ${change_not_near:+,.0f}')
print(f'DiD estimate:              ${DiD_manual:+,.0f}')
print(f'Interpretation: The incinerator announcement reduced nearby home prices by ${abs(DiD_manual):,.0f}')

### Step 3.3 — Visualise group means

In [ ]:
fig, ax = plt.subplots(figsize=(7,5))
ax.plot([1978,1981], [m_n0_78, m_n0_81], 'bs-', label='Control (not near)', linewidth=2)
ax.plot([1978,1981], [m_n1_78, m_n1_81], 'ro-', label='Treated (near)', linewidth=2)
counterfactual = m_n1_78 + (m_n0_81 - m_n0_78)
ax.plot([1978,1981], [m_n1_78, counterfactual], 'r--', alpha=0.5, label='Treated counterfactual')
ax.annotate('', xy=(1981, m_n1_81), xytext=(1981, counterfactual),
            arrowprops=dict(arrowstyle='<->', color='green', lw=2))
ax.text(1981.1, (m_n1_81+counterfactual)/2, f'DiD=${DiD_manual:,.0f}', color='green', va='center')
ax.set_xticks([1978, 1981])
ax.set_ylabel('Mean Real House Price ($)')
ax.set_title('DiD: Effect of Incinerator Announcement on House Prices')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(output_dir,'fig_ch13_01_did_plot.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4 — DiD via OLS Regression

The DiD estimate can be recovered via OLS:

$$rprice_i = \beta_0 + \beta_1 \cdot nearinc_i + \beta_2 \cdot y81_i + \delta \cdot (nearinc_i \times y81_i) + u_i$$

| | 1978 ($y81=0$) | 1981 ($y81=1$) | Change |
|---|---|---|---|
| Control | $\beta_0$ | $\beta_0+\beta_2$ | $\beta_2$ |
| Treated | $\beta_0+\beta_1$ | $\beta_0+\beta_1+\beta_2+\delta$ | $\beta_2+\delta$ |
| **DiD** | | | $\delta$ |

### Step 4.1 — Create interaction term manually

In [ ]:
df['nearinc_y81'] = df['nearinc'] * df['y81']
print('Dataset with interaction term (first 8 rows):')
print(df[['year','y81','nearinc','nearinc_y81','rprice']].head(8))

### Step 4.2 — Manual OLS: rprice ~ nearinc + y81 + nearinc_y81

In [ ]:
n = len(df)
X = np.column_stack([
    np.ones(n),
    df['nearinc'].values,
    df['y81'].values,
    df['nearinc_y81'].values
])
y_arr = df['rprice'].values
beta_manual = np.linalg.solve(X.T @ X, X.T @ y_arr)
labels = ['const','nearinc','y81','nearinc_y81']
print('Manual OLS (no controls):')
for lbl, b in zip(labels, beta_manual):
    print(f'  {lbl:15s}: {b:.2f}')

### Step 4.3 — Statsmodels OLS

In [ ]:
res_did = smf.ols('rprice ~ nearinc + y81 + nearinc_y81', data=df).fit()
print(res_did.summary())

### Step 4.4 — Compare manual vs statsmodels

In [ ]:
sm_p = res_did.params
print('Coefficient comparison:')
for lbl, b_man in zip(labels, beta_manual):
    b_sm = sm_p[lbl]
    match = np.isclose(b_man, b_sm)
    print(f'  {lbl:15s}: manual={b_man:.2f}, sm={b_sm:.2f}, match={match}')

### Step 4.5 — Verify DiD: OLS matches group-means

In [ ]:
did_ols = res_did.params['nearinc_y81']
match_did = np.isclose(DiD_manual, did_ols)
print(f'DiD (group means): ${DiD_manual:,.2f}')
print(f'DiD (OLS delta):   ${did_ols:,.2f}')
print(f'They match: {match_did}')

## 5 — DiD with Controls

Adding control variables can improve precision and reduce omitted variable bias.

The DiD coefficient $\delta$ should not change much if the parallel trends assumption holds,
but adding controls often reduces the standard error.

### Step 5.1 — Inspect available control variables

In [ ]:
print('Available columns:', list(df.columns))
print('\nFirst rows with potential controls:')
control_cols = [c for c in df.columns if c not in ['rprice','nearinc','y81','nearinc_y81','year']]
print(df[control_cols[:6]].head())

### Step 5.2 — Manual OLS with controls

In [ ]:
# Use age and age^2 as controls (house characteristics)
df['age2'] = df['age']**2
print('After adding age2, first 5 rows:')
print(df[['age','age2','rprice']].head())

In [ ]:
X_ctrl = np.column_stack([
    np.ones(n),
    df['nearinc'].values,
    df['y81'].values,
    df['nearinc_y81'].values,
    df['age'].values,
    df['age2'].values
])
beta_ctrl = np.linalg.solve(X_ctrl.T @ X_ctrl, X_ctrl.T @ y_arr)
labels_ctrl = ['const','nearinc','y81','nearinc_y81','age','age2']
print('Manual OLS with controls:')
for lbl, b in zip(labels_ctrl, beta_ctrl):
    print(f'  {lbl:15s}: {b:.4f}')

### Step 5.3 — Statsmodels with controls

In [ ]:
res_ctrl = smf.ols('rprice ~ nearinc + y81 + nearinc_y81 + age + age2', data=df).fit()
print(res_ctrl.summary())

### Step 5.4 — Compare manual vs statsmodels (with controls)

In [ ]:
sm_ctrl = res_ctrl.params
print('Coefficient comparison (with controls):')
for lbl, b_man in zip(labels_ctrl, beta_ctrl):
    b_sm = sm_ctrl[lbl]
    match = np.isclose(b_man, b_sm)
    print(f'  {lbl:15s}: manual={b_man:.2f}, sm={b_sm:.2f}, match={match}')

### Step 5.5 — Compare DiD with and without controls

In [ ]:
did_no_ctrl = res_did.params['nearinc_y81']
did_ctrl    = res_ctrl.params['nearinc_y81']
print(f'DiD without controls: ${did_no_ctrl:,.0f}  (SE={res_did.bse["nearinc_y81"]:,.0f})')
print(f'DiD with controls:    ${did_ctrl:,.0f}  (SE={res_ctrl.bse["nearinc_y81"]:,.0f})')
print('Controls reduce SE — more precise estimate')

## 6 — Log Specification

**Log model:** $\ln(rprice) = \beta_0 + \beta_1 nearinc + \beta_2 y81 + \delta(nearinc \times y81) + u$

$\delta$ now measures the **percentage** change in price due to the incinerator.

In [ ]:
df['lrprice'] = np.log(df['rprice'])
print('Log prices added (first 5 rows):')
print(df[['rprice','lrprice']].head())

In [ ]:
# Manual OLS for log model
y_log = df['lrprice'].values
X_log = np.column_stack([np.ones(n), df['nearinc'].values, df['y81'].values, df['nearinc_y81'].values])
beta_log = np.linalg.solve(X_log.T @ X_log, X_log.T @ y_log)
print('Manual OLS (log model):')
for lbl, b in zip(labels, beta_log):
    print(f'  {lbl:15s}: {b:.4f}')

In [ ]:
res_log = smf.ols('lrprice ~ nearinc + y81 + nearinc_y81', data=df).fit()
print(res_log.summary())

In [ ]:
sm_log = res_log.params
print('Log model comparison:')
for lbl, b_man in zip(labels, beta_log):
    b_sm = sm_log[lbl]
    print(f'  {lbl:15s}: manual={b_man:.4f}, sm={b_sm:.4f}, match={np.isclose(b_man,b_sm)}')
print(f'\nInterpretation: nearinc_y81 = {res_log.params["nearinc_y81"]:.4f}')
print(f'  = {100*res_log.params["nearinc_y81"]:.1f}% reduction in price for near-incinerator homes after announcement')

## 7 — Summary Table

In [ ]:
sumtab = pd.DataFrame({
    'Model': ['No controls (levels)','With controls (levels)','No controls (log)'],
    'DiD_coeff': [
        res_did.params['nearinc_y81'],
        res_ctrl.params['nearinc_y81'],
        res_log.params['nearinc_y81']
    ],
    'DiD_SE': [
        res_did.bse['nearinc_y81'],
        res_ctrl.bse['nearinc_y81'],
        res_log.bse['nearinc_y81']
    ],
    'DiD_pval': [
        res_did.pvalues['nearinc_y81'],
        res_ctrl.pvalues['nearinc_y81'],
        res_log.pvalues['nearinc_y81']
    ]
})
print(sumtab.round(4).to_string(index=False))
sumtab.to_csv(os.path.join(output_dir,'table_ch13_did_summary.csv'), index=False)
print('Saved table_ch13_did_summary.csv')